# Probability, sampling (temperature, top-k, top-p, min-p)

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Probability

> **Problem.** A support bot answers "the invoice total is 1,250" with the same confident tone whether the model was 99% sure or 51%. To route uncertain answers to a human, you need the model's actual confidence — which it has, at every token.

**Idea.** The model's output is a probability distribution over the next token; the API exposes it as log-probabilities.

**Use when** confidence thresholds, classification, detecting hedging, calibrating routing.  
**Not when** long free-form answers — per-token confidence does not summarise a paragraph.

```
prompt: "capital of France?"   ──model──▶  next token distribution

   Paris  ████████████████████████  0.9993
   Par    ▏                         0.0004
   The    ▏                         0.0002
   …
   log-probability −0.0007  ──exp──▶  0.9993
```

**How it works.**
1. `logprobs=True, top_logprobs=8` asks the API to return the 8 most likely next tokens with their log-probabilities.
2. `np.exp(logprob)` converts each to a probability; the top-8 sum to nearly 1 when the model is certain.
3. Entropy, `−Σ p·log p`, is one number for "how spread out" the distribution is: 0 means one certain token.
4. Note the tokens are subword pieces (`Par` + `is`), not words — probability lives at the token level.

| | what happens | result |
|:--|:--|:--|
| ✓ certain | "capital of France" | top token ≈ 0.999, entropy ≈ 0 |
| ✓ uncertain | an ambiguous question | mass spread over several tokens, entropy high |

**Production code and its real output**

In [2]:
# Probability — the model's output is a distribution over the next token. The API exposes the
# top candidates as log-probabilities; exp() turns them into probabilities.
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France? Answer with the city name only.",
        }
    ],
    max_tokens=1,
    logprobs=True,
    top_logprobs=8,
    temperature=0,
)
candidates = response.choices[0].logprobs.content[0].top_logprobs
rows = []
logprobs = []
for candidate in candidates:
    logprobs.append(candidate.logprob)
    rows.append(
        {
            "token": candidate.token,
            "logprob": round(candidate.logprob, 3),
            "prob": round(float(np.exp(candidate.logprob)), 6),
        }
    )
show("next-token distribution", rows)

probabilities = np.exp(np.array(logprobs))
entropy = float(-np.sum(probabilities * np.log(probabilities)))
print(
    "sum of top-8:",
    round(float(probabilities.sum()), 6),
    "| entropy:",
    round(entropy, 4),
    "nats (0 = certain)",
)
assert rows[0]["token"].strip().lower().startswith("par")

next-token distribution
[
  {
    "token": "Paris",
    "logprob": -0.0,
    "prob": 0.999999
  },
  {
    "token": " Paris",
    "logprob": -14.25,
    "prob": 1e-06
  },
  {
    "token": "Par",
    "logprob": -15.875,
    "prob": 0.0
  },
  {
    "token": "巴黎",
    "logprob": -16.0,
    "prob": 0.0
  },
  {
    "token": "PAR",
    "logprob": -16.375,
    "prob": 0.0
  },
  {
    "token": " paris",
    "logprob": -16.875,
    "prob": 0.0
  },
  {
    "token": " باريس",
    "logprob": -17.25,
    "prob": 0.0
  },
  {
    "token": "Berlin",
    "logprob": -17.625,
    "prob": 0.0
  }
]
sum of top-8: 1.0 | entropy: 0.0 nats (0 = certain)


**What the output shows.** `Paris` carried almost all the probability; the top-8 summed to ≈1 and entropy was near zero — the model was certain, and you can measure that.

**In practice**
- **classification** — for yes/no or label outputs, `max_tokens=1` + logprobs gives a calibrated score for free — better than parsing text.
- **thresholds** — route answers with top-token probability below a threshold to a fallback or a human; tune the threshold on labelled data.
- **hedging detection** — high entropy on the first content token predicts "I'm not sure"-style answers before they are written.
- **cost** — logprobs are free in tokens but add response size; request `top_logprobs` only where you use them.

**Alternatives** — asking the model to rate its own confidence (poorly calibrated) · sampling several answers and measuring agreement (self-consistency, layer 3)

**Terms** — *log-probability*: log of a probability; 0 = certain, more negative = less likely · *entropy*: a measure of uncertainty · *token*: a subword piece


### temperature

> **Problem.** The same prompt must give the same answer for an extraction job, but varied, creative answers for a brainstorming feature. Both run on one model; one setting has to control how adventurous the sampling is.

**Idea.** Divide the logits by a temperature before softmax: below 1 sharpens toward the top token, above 1 flattens toward randomness.

**Use when** extraction, classification, code → 0; conversation → 0.7; brainstorming → 1.0–1.5.  
**Not when** you rely on T=0 for reproducibility — it is deterministic-ish, not guaranteed (use `seed` too).

```
T = 0.2   ████████████░░░░░░░░   one token dominates      → same answer every time
T = 1.0   ██████░░░░░░░░░░░░░░   the model's own spread
T = 1.5   ████░░░░░░░░░░░░░░░░   flatter                  → varied, sometimes odd
```

**How it works.**
1. The cell takes a real top-20 distribution from the API and recomputes softmax at several temperatures — the same math the server runs.
2. At T=0.2 the top token holds most of the mass; at T=1.5 many tokens share it.
3. Five API calls at T=0 return the same word; five at T=1.5 return several different words.
4. Temperature changes which tokens are *likely*, not which are *possible* — unlike top-k / top-p (next items).

| | what happens | result |
|:--|:--|:--|
| ✓ T=0 | 5 calls | 1 distinct reply |
| ✓ T=1.5 | 5 calls | several distinct replies |
| ✗ T=2+ | very flat | incoherent tokens creep in |

**Production code and its real output**

In [3]:
# A real next-token distribution to sample from (top 20 candidates from the API).
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[{"role": "user", "content": "Write one word that describes the ocean."}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=20,
    temperature=0,
)
tokens = []
logits = []
for candidate in response.choices[0].logprobs.content[0].top_logprobs:
    tokens.append(candidate.token)
    logits.append(candidate.logprob)
logits = np.array(logits)
import torch  # noqa: E402


def distribution(temperature: float = 1.0) -> np.ndarray:
    return torch.softmax(torch.tensor(logits) / temperature, dim=0).numpy()


def top(probabilities: np.ndarray, n: int = 6) -> str:
    order = np.argsort(-probabilities)[:n]
    parts = []
    for index in order:
        if probabilities[index] > 0:
            parts.append(f"{tokens[index]!r}:{probabilities[index]:.3f}")
    return "  ".join(parts)


# Temperature — divides the logits: <1 sharpens, >1 flattens. The API applies the same math.
for temperature in [0.2, 0.7, 1.0, 1.5]:
    print(f"T={temperature:<4}", top(distribution(temperature)))

replies = {}
for temperature in [0.0, 1.5]:
    words = set()
    for _ in range(5):
        reply = client.chat.completions.create(
            model=settings.openai_model,
            messages=[{"role": "user", "content": "Write one word that describes the ocean."}],
            max_tokens=3,
            temperature=temperature,
        )
        words.add(reply.choices[0].message.content.strip().lower())
    replies[temperature] = sorted(words)
print("API T=0.0 → distinct replies:", replies[0.0])
print("API T=1.5 → distinct replies:", replies[1.5])
assert distribution(0.2).max() > distribution(1.5).max()

T=0.2  'V':1.000  'M':0.000  'Maj':0.000  'Myst':0.000  'Exp':0.000  ' Vast':0.000
T=0.7  'V':0.976  'M':0.013  'Maj':0.009  'Myst':0.001  'Exp':0.000  ' Vast':0.000
T=1.0  'V':0.909  'M':0.045  'Maj':0.035  'Myst':0.006  'Exp':0.001  ' Vast':0.001
T=1.5  'V':0.741  'M':0.100  'Maj':0.085  'Myst':0.026  'Exp':0.010  ' Vast':0.006


API T=0.0 → distinct replies: ['vast.']
API T=1.5 → distinct replies: ['vast.']


**What the output shows.** The recomputed distributions sharpen and flatten with T; the API calls at T=0 all agreed while T=1.5 produced a spread of words.

**In practice**
- **start at 0** — for anything parsed by code; raise it only when you want variety.
- **seed for reproducibility** — T=0 plus `seed` gives best-effort determinism; the provider still cannot promise bit-identical outputs across model updates.
- **do not combine blindly** — temperature with top-p both set aggressively compounds; tune one, hold the other at default.
- **evaluate at the setting you ship** — an eval run at T=0 says nothing about behaviour at T=0.9.

**Alternatives** — top-p / top-k / min-p (truncate the tail instead of reshaping it) · `seed` for repeatability

**Terms** — *temperature*: divisor applied to logits before softmax · *deterministic*: same input → same output


### top-k

> **Problem.** Even at moderate temperature, the long tail of the distribution — thousands of tokens with tiny probability — occasionally gets sampled, and one nonsense token derails a whole paragraph.

**Idea.** Keep only the k most likely tokens, renormalise, sample from those.

**Use when** local models where the parameter exists (llama.cpp, vLLM, HF generate); k≈40–50 as a safe default.  
**Not when** the OpenAI API — it does not expose top-k; use top-p there.

```
sorted probabilities   ████ ███ ██ ▌▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏
top_k = 3              ████ ███ ██ ─────────────── cut, renormalise
```

**How it works.**
1. Sort the distribution, keep the k largest (`torch.topk`), set the rest to zero.
2. Divide the survivors by their sum so they form a distribution again.
3. k=1 is greedy decoding; k=3 keeps three candidates; k=10 keeps ten regardless of how flat or peaked the distribution is.

| | what happens | result |
|:--|:--|:--|
| ✓ k=1 | greedy | always the top token |
| ✓ k=3 | 3 candidates | small, controlled variety |
| ✗ fixed k on a flat distribution | cuts good candidates | top-p adapts, top-k does not |

**Production code and its real output**

In [4]:
# A real next-token distribution to sample from (top 20 candidates from the API).
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[{"role": "user", "content": "Write one word that describes the ocean."}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=20,
    temperature=0,
)
tokens = []
logits = []
for candidate in response.choices[0].logprobs.content[0].top_logprobs:
    tokens.append(candidate.token)
    logits.append(candidate.logprob)
logits = np.array(logits)
import torch  # noqa: E402


def distribution(temperature: float = 1.0) -> np.ndarray:
    return torch.softmax(torch.tensor(logits) / temperature, dim=0).numpy()


def top(probabilities: np.ndarray, n: int = 6) -> str:
    order = np.argsort(-probabilities)[:n]
    parts = []
    for index in order:
        if probabilities[index] > 0:
            parts.append(f"{tokens[index]!r}:{probabilities[index]:.3f}")
    return "  ".join(parts)


# Top-k — keep only the k most likely tokens, renormalise. torch.topk does the selection.
def top_k(probabilities: np.ndarray, k: int) -> np.ndarray:
    values, indexes = torch.topk(torch.tensor(probabilities), k)
    kept = np.zeros_like(probabilities)
    kept[indexes.numpy()] = values.numpy()
    return kept / kept.sum()


base = distribution()
for k in [1, 3, 10]:
    filtered = top_k(base, k)
    print(f"top_k={k:<3} kept {int((filtered > 0).sum()):>2} tokens  ", top(filtered))
assert int((top_k(base, 3) > 0).sum()) == 3

top_k=1   kept  1 tokens   'V':1.000
top_k=3   kept  3 tokens   'V':0.928  'Maj':0.041  'M':0.032
top_k=10  kept 10 tokens   'V':0.919  'Maj':0.040  'M':0.031  'Myst':0.005  'Exp':0.001  'End':0.001


**What the output shows.** k=1 kept one token, k=3 kept three, k=10 kept ten — the count is fixed by k, not by the shape of the distribution.

**In practice**
- **k is blind to shape** — a peaked distribution needs k=2, a flat one k=100; a fixed k is wrong for one of them — prefer top-p or min-p.
- **combine with temperature** — typical local-model defaults: T=0.7, top_k=40, top_p=0.95 together.
- **greedy repeats** — k=1 (greedy) loops and repeats on long outputs; add a repetition penalty if you must use it.

**Alternatives** — top-p (adapts to the distribution) · min-p (adapts to confidence) · beam search for short exact outputs

**Terms** — *top-k*: keep the k most likely tokens · *greedy*: always pick the single most likely token · *renormalise*: rescale so probabilities sum to 1 again


### top-p

> **Problem.** A fixed top-k cuts too much when the model is unsure (many good candidates) and too little when it is confident (one good candidate plus junk). The cut-off needs to move with the distribution.

**Idea.** Keep the smallest set of tokens whose probabilities add up to p, then sample from that set.

**Use when** the default knob on most APIs; 0.9–0.95 for general text.  
**Not when** you need exact repeatability — set T=0 instead.

```
peaked   ████████ ██ ▏▏▏▏          p=0.7 → keeps 1–2 tokens
flat     ███ ███ ██ ██ ██ █ █      p=0.7 → keeps 5–6 tokens
```

**How it works.**
1. Sort probabilities descending and take the running total.
2. Keep tokens until the running total reaches p; drop the rest.
3. Renormalise the survivors and sample.
4. On a peaked distribution p=0.7 keeps very few tokens; on a flat one it keeps many — the same p adapts.

| | what happens | result |
|:--|:--|:--|
| ✓ peaked, p=0.7 | top token already ≥ 0.7 | keeps 1–2 |
| ✓ flat, p=0.7 | needs many tokens to reach 0.7 | keeps 5–6 |
| ✗ p=1.0 | keeps everything | the tail is back |

**Production code and its real output**

In [5]:
# A real next-token distribution to sample from (top 20 candidates from the API).
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[{"role": "user", "content": "Write one word that describes the ocean."}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=20,
    temperature=0,
)
tokens = []
logits = []
for candidate in response.choices[0].logprobs.content[0].top_logprobs:
    tokens.append(candidate.token)
    logits.append(candidate.logprob)
logits = np.array(logits)
import torch  # noqa: E402


def distribution(temperature: float = 1.0) -> np.ndarray:
    return torch.softmax(torch.tensor(logits) / temperature, dim=0).numpy()


def top(probabilities: np.ndarray, n: int = 6) -> str:
    order = np.argsort(-probabilities)[:n]
    parts = []
    for index in order:
        if probabilities[index] > 0:
            parts.append(f"{tokens[index]!r}:{probabilities[index]:.3f}")
    return "  ".join(parts)


# Top-p (nucleus) — keep the smallest set whose cumulative probability ≥ p. Adapts to the shape.
def top_p(probabilities: np.ndarray, p: float) -> np.ndarray:
    order = np.argsort(-probabilities)
    cumulative = np.cumsum(probabilities[order])
    keep = int(np.searchsorted(cumulative, p)) + 1
    kept = np.zeros_like(probabilities)
    for index in order[:keep]:
        kept[index] = probabilities[index]
    return kept / kept.sum()


peaked, flat = distribution(1.0), distribution(3.0)
for p in [0.3, 0.7, 0.95]:
    print(
        
            f"top_p={p:<5} peaked keeps {int((top_p(peaked, p) > 0).sum()):>2} | flat keeps "
            f"{int((top_p(flat, p) > 0).sum()):>2}"
        
    )
assert int((top_p(flat, 0.7) > 0).sum()) >= int((top_p(peaked, 0.7) > 0).sum())

top_p=0.3   peaked keeps  1 | flat keeps  1
top_p=0.7   peaked keeps  1 | flat keeps  5
top_p=0.95  peaked keeps  2 | flat keeps 16


**What the output shows.** With the same p, the peaked distribution kept a handful of tokens and the flattened one kept many more — the cut moved with the shape.

**In practice**
- **0.9–0.95** — the usual production range; 1.0 disables it.
- **with temperature** — raise one, not both; T=1.0 + p=0.9 or T=0.7 + p=1.0 are common pairs.
- **API naming** — OpenAI: `top_p`; Anthropic: `top_p`; local engines: `top_p` — the one sampling knob every provider shares.

**Alternatives** — min-p (scales with the top token) · top-k · typical sampling

**Terms** — *nucleus*: the kept set of tokens · *cumulative probability*: running total of sorted probabilities


### min-p

> **Problem.** Top-p still admits junk on peaked distributions: if the top token is 0.60 and p=0.95, dozens of near-zero tokens are needed to fill the remaining 0.35. A cut-off relative to the best token avoids that.

**Idea.** Keep only tokens whose probability is at least min_p × the top token's probability.

**Use when** local models (llama.cpp, vLLM) where quality at higher temperature matters; 0.05–0.1 is typical.  
**Not when** hosted APIs that do not expose it.

```
top token 0.60, min_p 0.1  →  floor 0.06  →  keep tokens ≥ 0.06   (few: confident)
top token 0.15, min_p 0.1  →  floor 0.015 →  keep tokens ≥ 0.015  (many: unsure)
```

**How it works.**
1. Find the top token's probability; the floor is `min_p` times that.
2. Keep every token at or above the floor, drop the rest, renormalise.
3. When the model is confident the floor is high and few tokens survive; when it is unsure the floor drops and more survive.

| | what happens | result |
|:--|:--|:--|
| ✓ min_p=0.5 | strict | 1–2 tokens |
| ✓ min_p=0.1 | usual | confident → few, unsure → many |
| ✓ min_p=0.02 | loose | most of the top 20 |

**Production code and its real output**

In [6]:
# A real next-token distribution to sample from (top 20 candidates from the API).
response = client.chat.completions.create(
    model=settings.openai_model,
    messages=[{"role": "user", "content": "Write one word that describes the ocean."}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=20,
    temperature=0,
)
tokens = []
logits = []
for candidate in response.choices[0].logprobs.content[0].top_logprobs:
    tokens.append(candidate.token)
    logits.append(candidate.logprob)
logits = np.array(logits)
import torch  # noqa: E402


def distribution(temperature: float = 1.0) -> np.ndarray:
    return torch.softmax(torch.tensor(logits) / temperature, dim=0).numpy()


def top(probabilities: np.ndarray, n: int = 6) -> str:
    order = np.argsort(-probabilities)[:n]
    parts = []
    for index in order:
        if probabilities[index] > 0:
            parts.append(f"{tokens[index]!r}:{probabilities[index]:.3f}")
    return "  ".join(parts)


# Min-p — keep tokens with probability ≥ min_p × the top token's probability. Scales with confidence.
def min_p(probabilities: np.ndarray, threshold: float) -> np.ndarray:
    kept = np.where(probabilities >= threshold * probabilities.max(), probabilities, 0.0)
    return kept / kept.sum()


peaked, flat = distribution(1.0), distribution(3.0)
for threshold in [0.5, 0.1, 0.02]:
    print(
        
            f"min_p={threshold:<5} peaked keeps {int((min_p(peaked, threshold) > 0).sum()):>2} | flat "
            f"keeps {int((min_p(flat, threshold) > 0).sum()):>2}"
        
    )
assert int((min_p(flat, 0.1) > 0).sum()) >= int((min_p(peaked, 0.1) > 0).sum())

min_p=0.5   peaked keeps  1 | flat keeps  1
min_p=0.1   peaked keeps  1 | flat keeps  5
min_p=0.02  peaked keeps  3 | flat keeps 20


**What the output shows.** The same threshold kept few tokens on the peaked distribution and more on the flattened one; the floor tracked the top token.

**In practice**
- **pairs with high T** — min-p is what makes T=1.2–1.5 usable on local models: variety without the junk tail.
- **local only** — OpenAI and Anthropic do not expose it; llama.cpp, vLLM, SGLang and HF `generate` do.
- **one truncation knob** — pick min-p *or* top-p; stacking them is confusing to tune.

**Alternatives** — top-p · top-k · typical sampling

**Terms** — *min-p*: a floor relative to the best token · *truncation sampling*: any method that drops the tail before sampling
